In [1]:
!pip install -q torch transformers datasets peft accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.8 MB/s eta 0:00:00


In [ ]:

# !pip install -q torch transformers datasets peft accelerate bitsandbytes sentencepiece

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ----------------
# CONFIG
# ----------------
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_LEN = 512
BATCH_SIZE = 4
EPOCHS = 3
LR = 2e-4

# ----------------
# 4-bit QLoRA config
# ----------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# ----------------
# Load tokenizer & model
# ----------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

model = prepare_model_for_kbit_training(model)

# ----------------
# LoRA setup
# ----------------
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ----------------
# Load dataset
# ----------------
dataset = load_dataset(
    "json",
    data_files={
        "train": "/content/train.jsonl"
    }
)

# ----------------
# Formatting function
# ----------------
def format_example(example):
    prompt = f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""

    tokens = tokenizer(
        prompt,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(
    format_example,
    remove_columns=dataset["train"].column_names
)

# ----------------
# Training args
# ----------------
training_args = TrainingArguments(
    output_dir="./qlora-output",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=1,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    logging_steps=50,
    save_steps=200,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none"
)

# ----------------
# Trainer
# ----------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

# ----------------
# Train
# ----------------
trainer.train()

# ----------------
# Save adapters only
# ----------------
model.save_pretrained("/content/adapters")
tokenizer.save_pretrained("/content/adapters")

print("✅ Training complete. Adapter saved to /content/adapters")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


Map:   0%|          | 0/1744 [00:00<?, ? examples/s]

Step,Training Loss
50,0.753354
100,0.658382
150,0.639692
200,0.663947
250,0.649861
300,0.678468
350,0.640890
400,0.648759
450,0.632307
500,0.601669


✅ Training complete. Adapter saved to /content/adapters
